# An image atlas of the tokens

A demo: UMAP of the tokens of many images at one block, twice:
- as a **scatter**: all tokens in gray (the photos, plus random crops of them for context), the chosen photo's
  tokens in colour (their concept), and the tokens drawn in the atlas outlined;
- as an **atlas**: the same map drawn with the **patches themselves**, each framed in its concept's colour.

Neighbouring patches are tokens the block finds similar. A slider (with `ipywidgets`) moves through the blocks,
from the input of block 1 (`b0`), and through the photos. Works with ViTs (16 px patches) and ResNets (32 px
cells of the last block's grid).

**Needs:** `pip install "nave-xai[animation]" ipywidgets`. **Downloads:** the default ViT-Ti/16's weights once (about 11 MB).

In [ ]:
import sys
if "google.colab" in sys.modules:       # on Google Colab: install what this notebook needs
    get_ipython().run_line_magic("pip", 'install -q "nave-xai[animation]" ipywidgets')
# elsewhere: pip install "nave-xai[animation]" ipywidgets
import numpy as np

MODEL, TRAINING = "vitt16", "imagenet"    # or "resnet50", ...
K = 8
N_IMAGES = 12
CROPS = 4                                  # random crops per photo, for context in the scatter
IMAGE = 2                                  # the photo in colour (2: the cat)
GRID = 28                                  # the atlas is GRID x GRID patches
UMAP_EPOCHS = 200
SEED = 0


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display


def match_colours(labels, reference, K):
    """Give every concept of `labels` the colour index of the reference concept it overlaps most
    (one-to-one, maximising the total overlap), so colours stay stable from frame to frame."""
    overlap = np.zeros((K, K))
    np.add.at(overlap, (labels.ravel(), reference.ravel()), 1)
    try:
        from scipy.optimize import linear_sum_assignment
        rows, cols = linear_sum_assignment(-overlap)
        mapping = dict(zip(rows, cols))
    except ImportError:                                     # greedy, without scipy
        mapping, used = {}, set()
        for a, b in sorted(np.ndindex(K, K), key=lambda ab: -overlap[ab]):
            if a not in mapping and b not in used:
                mapping[a] = b
                used.add(b)
    return np.vectorize(lambda a: mapping.get(a, a))(labels)


def knn_agreement(features, targets, k=5, n=1500, seed=0):
    """How well a point's k nearest neighbours (cosine) predict its target: leave-one-out accuracy."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(features), min(n, len(features)), replace=False)
    F = features[idx] / np.linalg.norm(features[idx], axis=1, keepdims=True)
    t = targets[idx]
    sim = F @ F.T
    np.fill_diagonal(sim, -np.inf)
    nearest = np.argsort(-sim, axis=1)[:, :k]
    votes = np.array([np.bincount(t[row], minlength=t.max() + 1).argmax() for row in nearest])
    return float((votes == t).mean())


def normalise_embedding(e):
    e = e - np.median(e, axis=0)
    return e / np.percentile(np.abs(e), 98)


def token_labels(maps, centres):
    """The concept at every patch centre of every image: one label per token."""
    return np.concatenate([m[np.ix_(centres, centres)].ravel() for m in maps])


def match_to(labels, reference, K):
    """Rename `labels` so each concept takes the id of the reference concept it overlaps most."""
    return match_colours(labels, reference, K)


def block_names(backbone):
    """The input of block 1 ("b0"), then every block."""
    return ["b0"] + list(backbone.blocks)


def pool_to(m, gh, gw):
    """Average a (C, h, w) map onto a gh x gw grid (each cell: the mean of the cells it covers)."""
    C, h, w = m.shape
    if (h, w) == (gh, gw):
        return m
    if h % gh == 0 and w % gw == 0:
        return m.reshape(C, gh, h // gh, gw, w // gw).mean((2, 4))
    from skimage.transform import resize
    return resize(m, (C, gh, gw), order=1, anti_aliasing=True, preserve_range=True)


def block_tokens(backbone, images, names):
    """For every block: the tokens of all images on the deepest block's grid, scaled to length 1.
    Returns (tokens per block, (grid height, grid width), cell size in pixels)."""
    mean, std = np.array(backbone.mean, np.float32), np.array(backbone.std, np.float32)
    per_block = {n: [] for n in names}
    grid = None
    for img in images:
        x = ((img / 255.0 - mean) / std).astype(np.float32).transpose(2, 0, 1)[None]
        out = backbone(x, names)
        if grid is None:
            grid = out[names[-1]].shape[2:]
        for n in names:
            t = pool_to(out[n][0], *grid)
            per_block[n].append(t.reshape(t.shape[0], -1).T)
    tokens = [np.concatenate(per_block[n]) for n in names]
    tokens = [t / (np.linalg.norm(t, axis=1, keepdims=True) + 1e-12) for t in tokens]
    return tokens, tuple(grid), images[0].shape[0] // grid[0]


def cell_centres(size, cell):
    return np.arange(cell // 2, size, cell)


def aligned_umap(features, epochs, window=2, regularisation=0.02):
    """UMAP of a sequence of point sets (the same points), each frame kept close to its neighbours."""
    a = umap.AlignedUMAP(n_neighbors=15, min_dist=0.2, n_epochs=epochs, alignment_regularisation=regularisation,
                         alignment_window_size=window)
    a.fit(features, relations=[{i: i for i in range(len(features[0]))}] * (len(features) - 1))
    return [normalise_embedding(e) for e in a.embeddings_]


def last_reference_umap(features, epochs):
    """UMAP of every frame, each starting from the last frame's layout."""
    last = umap.UMAP(n_neighbors=15, min_dist=0.2, n_epochs=epochs).fit_transform(features[-1])
    out = [umap.UMAP(n_neighbors=15, min_dist=0.2, n_epochs=epochs, init=last).fit_transform(f) for f in features[:-1]]
    return [normalise_embedding(e) for e in out + [last]]


def renamed_like(labels, reference, K):
    """A renaming of concept ids so each takes the id of the reference concept it overlaps most."""
    from scipy.optimize import linear_sum_assignment
    overlap = np.zeros((K, K))
    np.add.at(overlap, (labels, reference), 1)
    r, c = linear_sum_assignment(-overlap)
    rename = np.arange(K)
    rename[r] = c
    return rename

import warnings
import umap

warnings.filterwarnings("ignore", module="umap")               # UMAP's notices about threads and seeds
warnings.filterwarnings("ignore", category=FutureWarning)      # scikit-learn's, inside UMAP


In [ ]:
from skimage import data, transform

names = ["astronaut", "coffee", "chelsea", "rocket", "immunohistochemistry", "hubble_deep_field",
         "retina", "camera", "coins", "horse", "moon", "page"][:N_IMAGES]

def load(name):
    img = getattr(data, name)()
    img = np.repeat(img[..., None], 3, axis=2) if img.ndim == 2 else img[..., :3]
    return (transform.resize(img, (224, 224), anti_aliasing=False) * 255).astype(np.uint8)

images = [load(n) for n in names]
fig, axes = plt.subplots(1, len(images), figsize=(1.3 * len(images), 1.5))
for ax, im, n in zip(np.atleast_1d(axes), images, names):
    ax.imshow(im); ax.set_title(n, fontsize=7); ax.axis("off")
rng = np.random.default_rng(SEED)

def random_crop(img):
    s = int(224 * rng.uniform(0.4, 0.8)); y, x = rng.integers(0, 224 - s, 2)
    return (transform.resize(img[y:y + s, x:x + s], (224, 224), anti_aliasing=False) * 255).astype(np.uint8)

context = [random_crop(im) for im in images for _ in range(CROPS)]
everything = images + context                              # the photos first, then the crops
# your own photos: see the commented cell in 01_encoder_blocks.ipynb
print(len(images), "photos,", len(context), "crops")

In [ ]:
from nave_xai import NAVEImage

explainer = NAVEImage(MODEL, TRAINING, k=K, seed=SEED)
bb = explainer.backbone
names_b = block_names(bb)
explainer = NAVEImage(bb, layers=[names_b[-1]], k=K, seed=SEED).fit(everything)
tokens, grid, cell = block_tokens(bb, everything, names_b)
G = grid[0] * grid[1]
concept = token_labels([e.explanation for e in explainer.predict(everything)], cell_centres(224, cell))
palette = explainer.palette / 255
patches = np.stack([im[r:r + cell, c:c + cell] for im in everything
                    for r in range(0, grid[0] * cell, cell) for c in range(0, grid[1] * cell, cell)])
print(len(patches), "tokens,", cell, "px each")

## Scatter and atlas

UMAP places the tokens of one block; the plane is cut into a grid, and each atlas cell shows the patch whose
token lies closest to the cell's centre. Each block's UMAP is computed the first time it is shown.

In [ ]:
embeddings = {}

def embed(block):
    if block not in embeddings:
        e = umap.UMAP(n_neighbors=15, min_dist=0.1, n_epochs=UMAP_EPOCHS).fit_transform(tokens[block])
        embeddings[block] = (e - e.min(0)) / (e.max(0) - e.min(0) + 1e-9)
    return embeddings[block]


def show(block=len(names_b) - 1, image=IMAGE):
    e = embed(block)
    cells = np.minimum((e * GRID).astype(int), GRID - 1)
    best = {}
    for i, c in enumerate(map(tuple, cells)):
        d = np.sum((e[i] - (np.array(c) + 0.5) / GRID) ** 2)
        if c not in best or d < best[c][0]:
            best[c] = (d, i)
    shown = np.array([i for _, i in best.values()])
    canvas = np.ones((GRID * 18, GRID * 18, 3))
    for (cx, cy), (_, i) in best.items():
        y, x = (GRID - 1 - cy) * 18, cx * 18
        canvas[y:y + 18, x:x + 18] = palette[concept[i]]
        p = patches[i] if cell == 16 else transform.resize(patches[i], (16, 16), anti_aliasing=True, preserve_range=True)
        canvas[y + 1:y + 17, x + 1:x + 17] = p / 255
    mine = np.arange(image * G, (image + 1) * G)
    fig, (a0, a1, a2) = plt.subplots(1, 3, figsize=(17, 6), gridspec_kw=dict(width_ratios=[0.6, 1, 1]))
    a0.imshow(everything[image]); a0.set_title(f"photo: {names[image]}", fontsize=10); a0.axis("off")
    a1.scatter(*e.T, c="0.85", s=3, linewidths=0)
    a1.scatter(*e[shown].T, facecolors="none", edgecolors="0.35", s=10, linewidths=0.5)
    a1.scatter(*e[mine].T, c=palette[concept[mine]], s=22, edgecolors="black", linewidths=0.4)
    a1.set_xticks([]); a1.set_yticks([]); a1.set_aspect("equal")
    a1.set_title(f"{names_b[block]}: all tokens (gray), atlas patches (outlined), the photo (colour)", fontsize=9)
    a2.imshow(canvas); a2.axis("off"); a2.set_title(f"{names_b[block]}: atlas of {len(best)} patches", fontsize=10)
    plt.show()


show()

## Through the blocks and the photos

With `ipywidgets`, sliders move through the blocks and the photos. Without it, the cell draws a few blocks.

In [ ]:
try:
    import ipywidgets as widgets
    widgets.interact(show, block=widgets.IntSlider(len(names_b) - 1, 0, len(names_b) - 1, 1),
                     image=widgets.Dropdown(options=[(n, i) for i, n in enumerate(names)], value=IMAGE));
except ImportError:
    for b in sorted({0, len(names_b) // 2, len(names_b) - 1}):
        show(b)

## Things to look at

- what the early blocks group together (colour, texture, position) and what the last ones do;
- where the chosen photo's concepts sit among all the other tokens;
- your own photos, or another model.

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@inproceedings{boubekki2026nave,
  author    = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  editor    = {Ceci, Michelangelo and Kocev, Dragi and Monreale, Anna and Vens, Celine and Flach, Peter
               and Manco, Giuseppe and Spiliopoulou, Myra and Masciari, Elio},
  title     = {Multi-depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  booktitle = {Machine Learning and Knowledge Discovery in Databases. Research Track},
  publisher = {Springer Nature Switzerland},
  address   = {Cham},
  pages     = {449--467},
  year      = {2026},
  doi       = {10.1007/978-3-032-37664-0_26},
  isbn      = {978-3-032-37664-0}
}
```